In [ ]:
#define the model
from typing import Any

import torch
import torch.nn as nn

class NWkernelRegression(nn.Module):
    def __init__(self, **kwargs: Any) -> None:
        super().__init__( **kwargs)
        self.w = nn.Parameter(torch.rand((1,), requires_grad=True))

    def forward(self,queries,keys,values):
        queries = queries.repeat_interleave(keys.shape[1]).reshape((-1, keys.shape[1]))
        self.attention_weights = nn.functional.softmax(-((queries - keys) * self.w)**2 / 2, dim=1)
        return torch.bmm(self.attention_weights.unsqueeze(1),values.unsqueeze(-1)).reshape(-1)
    

#train

n_train = 50  
x_train, _ = torch.sort(torch.rand(n_train) * 5)
def f(x):
    return 2 * torch.sin(x) + x**0.8
y_train = f(x_train) + torch.normal(0.0, 0.5, (n_train,)) 

#n*n的矩阵
X_tile = x_train.repeat((n_train, 1))
Y_tile = y_train.repeat((n_train, 1))
#eye是单位矩阵I 为了掩码自己的数据类型
keys = X_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train, -1))
# n*（n-1）
values = Y_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train, -1))

net = NWkernelRegression()
loss = nn.MSELoss(reduction = 'none')
trainer = torch.optim.SGD(net.parameters(),lr = 0.5)
epochs = 5
for epoch in range(epochs):
    trainer.zero_grad()
    l = loss(net(x_train,keys,values), y_train)
    l.sum().backward()
    trainer.step()
    